# Figures 3 & 4 rerun — KS at σ = 0, seed 8 — Colab T4

Retrains **one cell of the v3 lognormal-κ sweep** (`sigma=0.0, seed=8` — the
run behind figure 6) and keeps the whole training trajectory, which that
sweep did not: it saved one rollout per cell, the trained policy's, while
figures 3 and 4 are about how the economy *changes* over training.

So the policy is evaluated at **30 training updates, spaced linearly**, 200
reset-free steps × 200 agents each, and every step of every one of those
rollouts is persisted. (jmbc's own diagnostics loop is log-spaced and cannot
be told otherwise, hence the dedicated script.)

Everything else is the sweep cell as it ran: `n_agents=200`, `num_envs=12`,
`total_timesteps=128000`, `max_steps=200`, `k_0 ~ U(10, 70)` redrawn per
episode, homogeneous κ. The protocol is declared in
`runs/ks-fig34-rerun/config.yaml` and re-asserted before training, so it
cannot drift.

One run, ~2 min on a T4. Output ≈ 10 MB.

Design writeup: `runs/ks-fig34-rerun/README.md`.

In [ ]:
# Setup: clone or update the repo, install (idempotent -- safe to re-run).
%cd /content
![ -d jax-marl-bc ] || git clone https://github.com/danmonuni/jax-marl-bc.git
%cd jax-marl-bc
!git pull
!pip install -q -r requirements.txt && pip install -q -e . --no-deps

In [ ]:
# Sanity: a GPU runtime is attached (Runtime > Change runtime type > T4 GPU).
!nvidia-smi -L

In [ ]:
# Mount Drive BEFORE the run so the result is saved as soon as it finishes.
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

def save_results(name='ks-fig34-rerun'):
    """Sync runs/<name>/results -> Drive (exact path, idempotent re-sync)."""
    src = f'runs/{name}/results'
    dst = f'/content/drive/MyDrive/jax-marl-bc-runs/{name}/results'
    assert os.path.exists(src), f"{src} missing - did the run finish?"
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"saved {src} -> {dst}")

## Verify the protocol before burning GPU time

One tiny CPU cell that prints what will actually run. Two things to check:

1. the `k_0` line reads `U(10, 70) per agent, resampled per_episode` — if it
   says `(constant)`, the override did not take and this would be
   `ks_n200_top`'s old initialization, not the sweep cell's;
2. the printed protocol matches the sweep's (`max_steps=200`,
   `num_envs=12`, ...). The smoke overrides below shrink
   `n_agents`/`num_envs`/`total_timesteps` only so it finishes in seconds.

`verify_protocol()` also asserts this before training and aborts on any
mismatch.

In [ ]:
!python runs/ks-fig34-rerun/rerun_fig34.py device=cpu n_snapshots=2 \
    protocol.env.n_agents=16 protocol.train.total_timesteps=2000 \
    protocol.train.num_envs=4 protocol.train.num_minibatches=2 \
    out_dir=/tmp/fig34_smoke 2>&1 | grep -E "k_0|economy|RECORD|^  env\.|^  train\."

## The run

Runs `runs/ks-fig34-rerun/config.yaml` as-is. Dotlist overrides go after the
script path; protocol entries are addressed nested:

```
!python runs/ks-fig34-rerun/rerun_fig34.py sigma=0.4 seed=3     # any other sweep cell
!python runs/ks-fig34-rerun/rerun_fig34.py n_snapshots=60       # denser training axis
!python runs/ks-fig34-rerun/rerun_fig34.py sim_steps=1000       # longer evaluations
!python runs/ks-fig34-rerun/rerun_fig34.py k_init_dist=constant # ks_n200_top's old k_0
```

In [ ]:
!python runs/ks-fig34-rerun/rerun_fig34.py

In [ ]:
save_results('ks-fig34-rerun')

## What was recorded

A quick look before pulling it down: the record's shape, that no auto-reset
landed inside an evaluation, and the two things figures 3 and 4 are made of —
the cross-sectional capital histogram (per-agent mean over the last 50 steps)
and mean wealth through training.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

RUN = 'runs/ks-fig34-rerun/results/ks/sigma_0.00_seed_8'
WINDOW = 50

with np.load(f'{RUN}/rollouts.npz') as z:
    ks, wealths = z['ks'], z['wealths']
    steps, done = z['snap_env_steps'], z['done']
    print('snapshots x steps x agents:', ks.shape)
    print('training env steps:', steps[0], '->', steps[-1])
    print('auto-resets inside the evals:', int(done.sum()), '(expect 0)')

k_bar = ks[:, -WINDOW:, :].mean(axis=1)          # [snapshots, agents]
w_bar = wealths[:, -WINDOW:, :].mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(k_bar[-1], bins=30, color='steelblue', alpha=0.8)
axes[0].set_xlabel(f'capital $k^i$ (mean of last {WINDOW} steps)')
axes[0].set_ylabel('agents'); axes[0].set_title('trained cross-section')
axes[1].plot(steps, w_bar.mean(axis=1), marker='o', ms=3)
axes[1].set_xlabel('training env steps'); axes[1].set_ylabel('mean wealth')
axes[1].set_title('through training')
plt.tight_layout(); plt.show()

## Then, locally

Pull `runs/ks-fig34-rerun/results/ks/sigma_0.00_seed_8/` from Drive into

```
runs/final-paper-runs/.rerun-final-fig34-sim/data/
```

and replot:

```bash
python runs/final-paper-runs/.rerun-final-fig34-sim/plot_fig34.py
# -> runs/final-paper-runs/3.png, 4.png
```